In [3]:
# Section 1 -- Imports
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.ensemble import IsolationForest

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

ML_FEATURES_PATH = (
    PROCESSED_DATA_DIR /
    "customer_features_ml.parquet"
)

CUSTOMER_FEATURES_PATH = (
    PROCESSED_DATA_DIR /
    "customer_features.parquet"
)

ml_df = pd.read_parquet(ML_FEATURES_PATH)
customer_features = pd.read_parquet(CUSTOMER_FEATURES_PATH)

feature_columns = [
    "recency_days",
    "log_frequency",
    "log_monetary",
    "log_average_order_value",
    "log_avg_items_per_order",
    "log_unique_products",
    "purchase_span_days",
    "return_order_share",
    "return_value_share",
    "has_return"
]

X = ml_df[feature_columns]

print("X shape:", X.shape)

X shape: (5878, 10)


In [4]:
# Section 2 — Isolation Forest
# Section 2 — Isolation Forest

isolation_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

anomaly_labels = isolation_forest.fit_predict(X)

anomaly_scores = isolation_forest.decision_function(X)

print("Anomalies:", (anomaly_labels == -1).sum())
print("Normal:", (anomaly_labels == 1).sum())

Anomalies: 294
Normal: 5584


In [5]:
# Section 3 — Customer Anomaly Results

anomaly_df = customer_features.copy()

anomaly_df["anomaly_label"] = anomaly_labels
anomaly_df["anomaly_score"] = anomaly_scores

anomaly_df["is_anomaly"] = (
    anomaly_df["anomaly_label"] == -1
)

print(
    "Anomaly percentage:",
    round(anomaly_df["is_anomaly"].mean() * 100, 2),
    "%"
)

Anomaly percentage: 5.0 %


In [6]:
# Section 4 — Anomaly Profile

anomaly_summary = (
    anomaly_df
    .groupby("is_anomaly")[
        [
            "recency_days",
            "frequency",
            "monetary",
            "average_order_value",
            "unique_products",
            "return_order_share",
            "return_value_share"
        ]
    ]
    .median()
    .round(2)
)

anomaly_summary

,recency_days,frequency,monetary,average_order_value,unique_products,return_order_share,return_value_share
is_anomaly,,,,,,,
False,92.11,3.0,867.74,278.29,46.0,0.0,0.0
True,276.57,2.0,835.60,314.57,11.0,0.3,0.1


In [8]:
# Section 5 — Save Results
ANOMALY_OUTPUT_PATH = (
    PROCESSED_DATA_DIR /
    "customer_anomalies.parquet"
)

anomaly_df.to_parquet(
    ANOMALY_OUTPUT_PATH,
    index=False
)

print("Saved:", ANOMALY_OUTPUT_PATH)

Saved: C:\Users\arman\OneDrive\Desktop\Data_Analyst\Projects\CustomerIQ_Unsupervised\CustomerIQ\data\processed\customer_anomalies.parquet
